In [ ]:
import pandas as pd

customer = pd.read_csv("/content/customer.csv")
demographic = pd.read_csv("/content/demographic.csv")
address = pd.read_csv("/content/address.csv")


In [ ]:
print(customer.columns)
print(demographic.columns)
print(address.columns)


Index(['INDIVIDUAL_ID', 'ADDRESS_ID', 'CURR_ANN_AMT', 'DAYS_TENURE',
       'CUST_ORIG_DATE', 'AGE_IN_YEARS', 'DATE_OF_BIRTH',
       'SOCIAL_SECURITY_NUMBER'],
      dtype='object')
Index(['INDIVIDUAL_ID', 'INCOME', 'HAS_CHILDREN', 'LENGTH_OF_RESIDENCE',
       'MARITAL_STATUS', 'HOME_MARKET_VALUE', 'HOME_OWNER', 'COLLEGE_DEGREE',
       'GOOD_CREDIT'],
      dtype='object')
Index(['ADDRESS_ID', 'LATITUDE', 'LONGITUDE', 'STREET_ADDRESS', 'CITY',
       'STATE', 'COUNTY'],
      dtype='object')


MERGE CUSTOMER + DEMOGRAPHIC

What we are doing:
Attach demographic details (income, credit, family status) to each customer.

Why:
Demographics drive risk, churn, and pricing decisions.

In [ ]:
cust_demo = customer.merge(
    demographic,
    on="INDIVIDUAL_ID",
    how="left"
)


In [ ]:
cust_demo.head()
cust_demo.shape


(2280321, 16)

In [ ]:
df_customer = cust_demo.merge(
    address,
    on="ADDRESS_ID",
    how="left"
)


In [ ]:
df_customer.shape
df_customer.head()


,INDIVIDUAL_ID,ADDRESS_ID,CURR_ANN_AMT,DAYS_TENURE,CUST_ORIG_DATE,AGE_IN_YEARS,DATE_OF_BIRTH,SOCIAL_SECURITY_NUMBER,INCOME,HAS_CHILDREN,...,HOME_MARKET_VALUE,HOME_OWNER,COLLEGE_DEGREE,GOOD_CREDIT,LATITUDE,LONGITUDE,STREET_ADDRESS,CITY,STATE,COUNTY
0,2.213000e+11,5.213000e+11,818.877997,1454.0,2018-12-09,44.474,1978-06-23,608-XX-7640,22500.0,1.0,...,50000 - 74999,1.0,1.0,1.0,32.578829,-96.305006,52966 Welch Crescent,Kaufman,TX,Kaufman
1,2.213001e+11,5.213001e+11,974.199182,1795.0,2018-01-02,72.559,1950-05-30,342-XX-6908,27500.0,0.0,...,50000 - 74999,1.0,0.0,0.0,32.732209,-97.000893,46887 Lawrence Green,Grand Prairie,TX,Dallas
2,2.213007e+11,5.213002e+11,967.375112,4818.0,2009-09-23,55.444,1967-07-07,240-XX-9224,42500.0,0.0,...,75000 - 99999,1.0,0.0,0.0,32.819777,-96.846938,787 Daniel Mews Suite 806,Dallas,TX,Dallas
3,2.213016e+11,5.213006e+11,992.409561,130.0,2022-07-25,53.558,1969-05-25,775-XX-6249,125000.0,1.0,...,175000 - 199999,1.0,0.0,1.0,32.684065,-97.162180,9846 Shaw Manor Apt. 774,Arlington,TX,Tarrant
4,2.213016e+11,5.213006e+11,784.633494,5896.0,2006-10-11,50.220,1972-09-25,629-XX-7298,87500.0,1.0,...,225000 - 249999,1.0,1.0,1.0,32.751398,-97.376745,431 David Port Apt. 164,Fort Worth,TX,Tarrant


CREATE TENURE (LOYALTY)

What we are doing:
Convert tenure from days to years.

Why:
Tenure in years is easier to understand and is a key churn & loyalty driver.

In [ ]:
df_customer["TENURE_YEARS"] = df_customer["DAYS_TENURE"] / 365


In [ ]:
df_customer[["DAYS_TENURE", "TENURE_YEARS"]].head()


,DAYS_TENURE,TENURE_YEARS
0,1454.0,3.983562
1,1795.0,4.917808
2,4818.0,13.200000
3,130.0,0.356164
4,5896.0,16.153425


CREATE TENURE BAND

What we are doing:
Group customers into loyalty categories.

Why:
Executives think in segments, not raw numbers.

In [ ]:
df_customer["TENURE_BAND"] = pd.cut(
    df_customer["TENURE_YEARS"],
    bins=[0, 1, 3, 5, 50],
    labels=["New", "Early", "Mid", "Loyal"]
)


In [ ]:
df_customer[["TENURE_YEARS", "TENURE_BAND"]].head()


,TENURE_YEARS,TENURE_BAND
0,3.983562,Mid
1,4.917808,Mid
2,13.200000,Loyal
3,0.356164,New
4,16.153425,Loyal


CREATE AGE BAND

What we are doing:
Group customers by age.

Why:
Age strongly affects risk, claims, and churn in insurance.

In [ ]:
df_customer["AGE_BAND"] = pd.cut(
    df_customer["AGE_IN_YEARS"],
    bins=[0, 25, 40, 60, 100],
    labels=["Young", "Mid", "Senior", "Elder"]
)


In [ ]:
df_customer[["AGE_IN_YEARS", "AGE_BAND"]].head()


,AGE_IN_YEARS,AGE_BAND
0,44.474,Senior
1,72.559,Elder
2,55.444,Senior
3,53.558,Senior
4,50.220,Senior


CREATE INCOME BAND

What we are doing:
Group customers by income level.

Why:
Income affects price sensitivity, churn, and product uptake.

In [ ]:
df_customer["INCOME_BAND"] = pd.qcut(
    df_customer["INCOME"],
    3,
    labels=["Low", "Medium", "High"]
)


In [ ]:
df_customer[["INCOME", "INCOME_BAND"]].head()


,INCOME,INCOME_BAND
0,22500.0,Low
1,27500.0,Low
2,42500.0,Low
3,125000.0,High
4,87500.0,Medium


CREATE PREMIUM BAND

What we are doing:
Segment customers by how much premium they pay.

Why:
Premium size = revenue tier and pricing power.

In [ ]:
df_customer["PREMIUM_BAND"] = pd.qcut(
    df_customer["CURR_ANN_AMT"],
    3,
    labels=["Low", "Medium", "High"]
)


In [ ]:
df_customer[["CURR_ANN_AMT", "PREMIUM_BAND"]].head()


,CURR_ANN_AMT,PREMIUM_BAND
0,818.877997,Low
1,974.199182,Medium
2,967.375112,Medium
3,992.409561,Medium
4,784.633494,Low


CREATE CREDIT RISK FLAG

What we are doing:
Standardise credit quality into a clean risk indicator.

Why:
Credit quality affects claims risk, churn, and pricing.

In [ ]:
df_customer["CREDIT_RISK"] = df_customer["GOOD_CREDIT"].map(
    {1: "Good", 0: "Poor"}
)


In [ ]:
df_customer[["GOOD_CREDIT", "CREDIT_RISK"]].head()


,GOOD_CREDIT,CREDIT_RISK
0,1.0,Good
1,0.0,Poor
2,0.0,Poor
3,1.0,Good
4,1.0,Good


CREATE FAMILY FLAG

What we are doing:
Identify whether a customer has children.

Why:
Families typically:

have higher retention

different risk patterns

different pricing sensitivity

In [ ]:
df_customer["FAMILY_STATUS"] = df_customer["HAS_CHILDREN"].map(
    {1: "Has_Children", 0: "No_Children"}
)


In [ ]:
df_customer[["HAS_CHILDREN", "FAMILY_STATUS"]].head()


,HAS_CHILDREN,FAMILY_STATUS
0,1.0,Has_Children
1,0.0,No_Children
2,0.0,No_Children
3,1.0,Has_Children
4,1.0,Has_Children


CREATE HOME OWNERSHIP FLAG

What we are doing:
Standardise home ownership into a clean category.

Why:
- Home owners are usually:
- more stable
- lower churn
- different risk profile

In [ ]:
df_customer["HOME_STATUS"] = df_customer["HOME_OWNER"].map(
    {1: "Home_Owner", 0: "Renter"}
)


In [ ]:
df_customer[["HOME_OWNER", "HOME_STATUS"]].head()


,HOME_OWNER,HOME_STATUS
0,1.0,Home_Owner
1,1.0,Home_Owner
2,1.0,Home_Owner
3,1.0,Home_Owner
4,1.0,Home_Owner


SAVE PHASE 2 OUTPUT

What we are doing:
Save the fully engineered customer feature table.

Why:
This file will be used for:

- segmentation
- pricing models
- churn analysis
- Power BI dashboards

In [ ]:
df_customer.to_csv(
    "/content/customer_features_phase2.csv",
    index=False
)
